# SIX-SENSE LoRA 파인튜닝 (Colab) — 정리본
Llama / Qwen을 **같은 데이터·같은 설정**으로 학습하고, 여러 평가셋으로 비교합니다. 경상도 발화 → 호출 형식(`<six_0>(object=1, type=1)<six_end>` 등).

## 사용법 (위에서 아래로 한 번씩)
1. **0. 설치** 셀은 세션을 새로 열었을 때 한 번만 실행합니다. 처음 설치한 뒤에는 **런타임 → 세션 다시 시작**을 한 번 하세요(torchao와 peft 충돌 방지). 그 뒤에는 설치 셀을 건너뛰고 **1. 설정**부터 실행합니다.
2. **1. 설정**에서 `MODEL_KEY`(`'llama'`/`'qwen'`), `VERSION`, `RUN_TRAIN`만 확인합니다.
   - `RUN_TRAIN = True`: 처음부터 학습 → 평가 → 저장
   - `RUN_TRAIN = False`: 이미 저장된 어댑터를 불러와 평가만 (학습 안 함, 세션이 끊긴 뒤 평가만 다시 볼 때)
3. Llama는 허깅페이스 사용 동의가 필요합니다(https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct). 동의 후 토큰을 Colab 왼쪽 🔑 Secrets에 `HF_TOKEN`으로 등록하세요.
4. 런타임 유형은 GPU(T4/L4)로 합니다.
5. Llama로 한 번, Qwen으로 한 번 실행한 뒤 마지막 **비교표** 셀을 실행하면 모델 비교가 나옵니다.

## 데이터 (구글 드라이브 `data_v2` 폴더)
`train_v2_*.jsonl`(**1개만**), `val_v2_*.jsonl`, `eval_*v2.jsonl`, 그리고 평가용 `eval_v3_*`, `eval_stt_aug.jsonl`. 학습 파일은 `train_v2_`로 시작하는 파일이 둘 이상이면 오류로 멈춥니다.

## 0. 설치 (새 세션에서 한 번만)

In [ ]:
!nvidia-smi
!pip -q install -U transformers peft accelerate datasets torchao
# 처음 설치했다면: 런타임 → 세션 다시 시작 후, 이 셀은 건너뛰고 1번부터 실행

## 1. 설정

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # torch를 불러오기 전에 설정해야 함

# ================= 여기만 바꾸세요 =================
MODEL_KEY = 'llama'      # 'llama' 또는 'qwen'
VERSION   = 'v2dict2'    # 저장 폴더 이름에 붙는 버전. 이전 결과(v2, v2dict)를 덮어쓰지 않도록 새 이름 사용
RUN_TRAIN = True         # True: 학습부터 / False: 저장된 어댑터로 평가만
USE_BF16  = False        # T4는 False(fp16). Qwen에서 loss가 nan이면 True로
EPOCHS    = 3
SEED      = 42
# ===================================================

ROOT = '/content/drive/MyDrive/six-sense'
MODELS = {'llama': 'meta-llama/Llama-3.2-1B-Instruct', 'qwen': 'Qwen/Qwen3.5-2B'}
BASE = MODELS[MODEL_KEY]
OUT = f'{ROOT}/runs/lora-{MODEL_KEY}-{VERSION}'      # 폴더는 드라이브 연결 뒤(2번 셀)에 만듭니다
print('모델:', BASE, '| 저장 폴더:', OUT, '| 학습:', RUN_TRAIN)

## 2. 공통 준비 (드라이브 · 데이터 경로 · 상수)

In [ ]:
import re, json, glob, math, time, random, collections, unicodedata, fnmatch, csv
import torch, pandas as pd
from google.colab import drive, userdata
drive.mount('/content/drive')
os.makedirs(OUT, exist_ok=True)      # 드라이브가 연결된 뒤에 저장 폴더 생성
try: os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as e: print('HF_TOKEN 없음(Qwen은 없어도 될 수 있음):', e)
random.seed(SEED); torch.manual_seed(SEED)
print('GPU:', torch.cuda.get_device_name(0), '| bf16 지원:', torch.cuda.is_bf16_supported())

# data_v2 폴더 위치를 자동으로 찾음 (train_v2_로 시작하는 파일은 정확히 1개여야 함)
_tr = glob.glob('/content/drive/MyDrive/data_v2/train_v2_*.jsonl') + glob.glob('/content/drive/MyDrive/*/data_v2/train_v2_*.jsonl')
if len(_tr) != 1:
    raise RuntimeError(f'train_v2_*.jsonl 이 {len(_tr)}개입니다(1개여야 함). 옛 파일은 이름을 "백업_..."으로 바꾸세요: {_tr}')
DATA = os.path.dirname(_tr[0])

def find(pat):
    """DATA 폴더에서 이름 패턴으로 파일 찾기 (한글 파일명 정규화 포함). 없으면 None"""
    for root, _, files in os.walk(DATA):
        for f in files:
            if fnmatch.fnmatch(unicodedata.normalize('NFC', f), pat): return os.path.join(root, f)
    return None
def load(path):
    with open(path, encoding='utf-8') as f: return [json.loads(l) for l in f if l.strip()]

SYSTEM = '당신은 스마트홈 제어 비서입니다. 사용자의 사투리와 모호한 표현을 이해해 기능 호출 형식으로만 답하세요.'
SPEC_PROMPT = SYSTEM + """
출력은 아래 호출 중에서만 고르고, 설명 없이 호출만 쓰세요. 여러 동작이면 말한 순서대로 이어 붙입니다.
기기 번호: 1 조명, 2 에어컨, 3 블라인드
<six_0>(object=1, type=1)<six_end> 조명 켜기 / type=2 끄기
<six_1>(object=1, type=1)<six_end> 조명 더 밝게 / type=2 은은하게
<six_0>(object=2, type=1)<six_end> 에어컨 켜기 / type=2 끄기
<six_1>(object=2, type=1)<six_end> 에어컨 온도 올리기 / type=2 내리기 / type=3 바람 세게 / type=4 바람 약하게
<six_0>(object=3, type=1)<six_end> 블라인드 올리기 / type=2 내리기 / type=3 멈추기 / type=4 반쯤 내리기
<six_0>(object=-1)<six_end> 어떤 기기인지 불분명할 때(되묻기)
<six_2>()<six_end> 조명·에어컨·블라인드와 관계없는 말(해당 없음)"""
VALID = re.compile(r'^(?:<six_0>\(object=[1-3], type=[1-4]\)<six_end>|<six_1>\(object=[1-3], type=[1-4]\)<six_end>|<six_0>\(object=-1\)<six_end>|<six_2>\(\)<six_end>)+$')
CTRL = re.compile(r'<six_[01]>\(object=[1-3]')      # 기기를 실제로 조작하는 호출
LEAD = re.compile(r'^((?:<six_[012]>\([^)]*\)<six_end>)+)')
print('데이터 폴더:', DATA)

## 3. 데이터 불러오기

In [ ]:
train = load(glob.glob(f'{DATA}/train_v2_*.jsonl')[0])
val   = load(glob.glob(f'{DATA}/val_v2_*.jsonl')[0])
ev    = load(glob.glob(f'{DATA}/eval_*v2.jsonl')[0])
ev_clean = [r for r in ev if not r.get('needs_review')]    # 사람이 확인할 사유가 없는 문장만 평가에 사용
print('학습', len(train), '| 검증', len(val), '| 50대 평가', len(ev), '→ 사용', len(ev_clean))
print('학습 유형:', dict(collections.Counter(r.get('category') for r in train)))

# 학습/검증과 50대 평가 문장이 겹치지 않는지 확인
norm = lambda t: re.sub(r'\s+', '', t)
overlap = {norm(r['text']) for r in train + val} & {norm(r['text']) for r in ev}
print('학습·검증과 50대 평가가 겹치는 문장:', len(overlap))

## 4. 모델 · 토크나이저 · LoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = '<|finetune_right_pad_id|>' if '<|finetune_right_pad_id|>' in tok.get_vocab() else tok.eos_token
EOT = tok.eos_token_id          # 모델마다 대화 종료 토큰이 다름(Llama <|eot_id|>, Qwen <|im_end|>)
END = tok.eos_token
dtype = torch.bfloat16 if USE_BF16 else torch.float16
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=dtype, device_map={'': 0})
N_PARAMS = sum(p.numel() for p in base.parameters()) / 1e9
print(BASE, '| 종료 토큰:', END, f'| 파라미터 {N_PARAMS:.2f}B')

if RUN_TRAIN:
    lcfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules='all-linear')
    model = get_peft_model(base, lcfg)
    for p in model.parameters():          # 학습되는 LoRA 가중치는 fp32로 (T4의 fp16 학습 안정성)
        if p.requires_grad: p.data = p.data.float()
    model.print_trainable_parameters()
else:
    model = PeftModel.from_pretrained(base, OUT)      # 저장된 어댑터 불러오기
    model.eval()
    print('저장된 어댑터를 불러왔습니다:', OUT)

## 5. 공통 함수 (생성 · 채점)

In [ ]:
@torch.no_grad()
def generate(rows, system=SYSTEM, bs=32, adapter=True):
    """adapter=False면 LoRA를 끄고 베이스 모델로 생성"""
    model.eval(); tok.padding_side = 'left'; preds = []
    for i in range(0, len(rows), bs):
        chunk = rows[i:i+bs]
        prompts = [tok.apply_chat_template([{'role':'system','content':system},{'role':'user','content':r['text']}],
                                           tokenize=False, add_generation_prompt=True, enable_thinking=False) for r in chunk]
        enc = tok(prompts, return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
        ctx = torch.autocast('cuda', dtype=dtype) if adapter else model.disable_adapter()
        with ctx:
            out = model.generate(**enc, max_new_tokens=80, do_sample=False, eos_token_id=EOT, pad_token_id=tok.pad_token_id)
        preds += [tok.decode(o[enc['input_ids'].shape[1]:], skip_special_tokens=True).strip() for o in out]
    return preds

def score(rows, preds, name):
    n = len(rows); em = fmt = mis = nn = 0; bycat = collections.defaultdict(lambda: [0, 0])
    for r, p in zip(rows, preds):
        cat = r.get('category') or ('복합' if len(r['functions']) > 1 else '단일')
        ok = (p == r['output']); em += ok; fmt += bool(VALID.match(p))
        bycat[cat][0] += ok; bycat[cat][1] += 1
        if r['output'] == '<six_2>()<six_end>':
            nn += 1; mis += bool(CTRL.search(p))
    print(f'[{name}] n={n}  의도 일치(정확 일치) {em/n:.1%} | 형식 준수 {fmt/n:.1%} | 오작동률 {mis/max(nn,1):.1%} ({mis}/{nn})')
    print('   유형별:', {k: f'{a/b:.0%}({b})' for k, (a, b) in sorted(bycat.items())})
    return dict(name=name, n=n, exact=em/n, fmt=fmt/n, misop=mis/max(nn, 1))

def lenient(rows, preds, name):
    ok = 0
    for r, p in zip(rows, preds):
        m = LEAD.match(p.strip()); ok += bool(m) and m.group(1) == r['output']
    print(f'[{name}] 앞부분 호출만 맞으면 인정: {ok/len(rows):.1%} ({ok}/{len(rows)})')

# ---- 결과 기록 (results.json에 같은 이름은 덮어써서 중복 방지) ----
RES_PATH = f'{OUT}/results.json'
if os.path.exists(RES_PATH):
    _d = json.load(open(RES_PATH, encoding='utf-8')); META, results = _d.get('meta', {}), _d.get('results', [])
else:
    META, results = {}, []
def record(res):
    global results
    results = [r for r in results if r['name'] != res['name']] + [res]
    json.dump(dict(meta=META, results=results), open(RES_PATH, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

## 6. 파인튜닝 전 성능 (비교군: 베이스 모델 + 호출 목록을 프롬프트로 알려준 경우)
학습할 때만 실행됩니다. 어댑터를 끈 상태로 평가합니다.

In [ ]:
if RUN_TRAIN:
    pb_val = generate(val, SPEC_PROMPT, adapter=False);      record(score(val, pb_val, '학습 전 / val'))
    pb_ev  = generate(ev_clean, SPEC_PROMPT, adapter=False); record(score(ev_clean, pb_ev, '학습 전 / 50대 평가'))
    lenient(val, pb_val, '학습 전 / val'); lenient(ev_clean, pb_ev, '학습 전 / 50대 평가')
else:
    print('RUN_TRAIN=False → 건너뜀')

## 7. 학습

In [ ]:
if RUN_TRAIN:
    from torch.utils.data import Dataset
    from transformers import Trainer, TrainingArguments
    MAXLEN = 192
    class SFT(Dataset):
        def __init__(self, rows):
            self.items = []
            for r in rows:
                prompt = tok.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':r['text']}],
                                                 tokenize=False, add_generation_prompt=True, enable_thinking=False)
                p = tok(prompt, add_special_tokens=False)['input_ids']
                a = tok(r['output'] + END, add_special_tokens=False)['input_ids']
                ids = (p + a)[:MAXLEN]; lab = ([-100]*len(p) + a)[:MAXLEN]      # 정답 부분에만 손실 적용
                self.items.append((ids, lab))
        def __len__(self): return len(self.items)
        def __getitem__(self, i): return self.items[i]
    def collate(b):
        m = max(len(x[0]) for x in b)
        ids = torch.full((len(b), m), tok.pad_token_id); lab = torch.full((len(b), m), -100); att = torch.zeros((len(b), m), dtype=torch.long)
        for i, (x, y) in enumerate(b):
            ids[i, :len(x)] = torch.tensor(x); lab[i, :len(y)] = torch.tensor(y); att[i, :len(x)] = 1
        return {'input_ids': ids, 'labels': lab, 'attention_mask': att}
    args = TrainingArguments(output_dir=f'{OUT}/ckpt', num_train_epochs=EPOCHS, per_device_train_batch_size=4, per_device_eval_batch_size=8,
        gradient_accumulation_steps=4, learning_rate=2e-4, lr_scheduler_type='cosine', warmup_steps=20, weight_decay=0.0,
        logging_steps=20, eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
        bf16=USE_BF16, fp16=not USE_BF16, report_to='none', remove_unused_columns=False, seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=SFT(train), eval_dataset=SFT(val), data_collator=collate)
    t0 = time.time(); trainer.train(); TRAIN_SEC = time.time() - t0
    model.save_pretrained(OUT); tok.save_pretrained(OUT)
    print('LoRA 어댑터 저장:', OUT)
else:
    print('RUN_TRAIN=False → 건너뜀')

## 8. 파인튜닝 후 성능 (val · 50대 평가)

In [ ]:
pa_val = generate(val);      record(score(val, pa_val, '학습 후 / val'))
pa_ev  = generate(ev_clean); record(score(ev_clean, pa_ev, '학습 후 / 50대 평가'))
if RUN_TRAIN:
    t0 = time.time(); _ = generate(val[:64]); SEC_PER = (time.time() - t0) / 64
    META = dict(model=BASE, key=MODEL_KEY, version=VERSION, train_rows=len(train), params_B=round(N_PARAMS, 2), train_sec=round(TRAIN_SEC),
                gen_sec_per_sample=round(SEC_PER, 3), gpu=torch.cuda.get_device_name(0))
    record(results[-1]); print(META)
print('\n[50대 평가에서 틀린 문장]')
for r, p in zip(ev_clean, pa_ev):
    if p != r['output']: print(f"- {r['text']} | 정답 {' + '.join(r['functions'])} | 출력 {p}")

## 9. 20·30대 평가 (v3)

In [ ]:
_p = find('eval_v3_20_30s.jsonl')
if _p:
    ev3 = [r for r in load(_p) if not r.get('needs_review')]
    print('v3 평가 문장:', len(ev3))
    pa_v3 = generate(ev3)
    record(score(ev3, pa_v3, '학습 후 / 20·30대 v3')); lenient(ev3, pa_v3, '학습 후 / 20·30대 v3')
    print('\n[틀린 문장]')
    for r, q in zip(ev3, pa_v3):
        if q != r['output']: print(f"- {r['text']} | 정답 {r['output']} | 출력 {q}")
else:
    print('eval_v3_20_30s.jsonl 없음 → 건너뜀'); ev3 = []

## 10. "해당 없음" 문장 오작동 평가
관계없는 말에 기기를 조작하면 `조작 오류`입니다. 이 값이 올라가지 않는지 가장 중요하게 봅니다.

In [ ]:
_p = find('eval_v3_해당없음_50대검수.jsonl')
if _p:
    na_rows = [r for r in load(_p) if not r.get('needs_review')]
    na_preds = generate(na_rows)
    bad = []
    for r, p in zip(na_rows, na_preds):
        r['kind'] = '해당 없음(정답)' if p == r['output'] else ('되묻기' if 'object=-1' in p else '조작 오류')
        if r['kind'] != '해당 없음(정답)': bad.append((r['kind'], r['text'], p))
    print('평가 문장:', len(na_rows), dict(collections.Counter(r['kind'] for r in na_rows)))
    _ce = sum(r['kind']=='조작 오류' for r in na_rows)
    print(f"조작 오류율: {_ce}/{len(na_rows)}")
    record(dict(name='학습 후 / 해당 없음 50대검수', n=len(na_rows), exact=sum(r['kind']=='해당 없음(정답)' for r in na_rows)/len(na_rows), fmt=1.0, misop=_ce/len(na_rows)))
    for b in bad: print(b)
else:
    print('eval_v3_해당없음_50대검수.jsonl 없음 → 건너뜀')

## 10-2. "해당 없음" 신규 평가 — 학습에 쓰지 않은 물건
`eval_해당없음_신규물건.jsonl`(전자레인지·에어프라이어·노트북 등)이 `data_v2`에 있으면 평가합니다. 보강 데이터가 **처음 보는 물건**에도 통하는지 보는 평가입니다.

In [ ]:
_p = find('eval_해당없음_신규물건.jsonl')
if _p:
    nw = load(_p); nw_preds = generate(nw)
    nw_bad = [(r['text'], q) for r, q in zip(nw, nw_preds) if q != r['output']]
    nw_ctrl = sum(bool(CTRL.search(q)) for q in nw_preds)
    print(f'신규 물건 해당 없음: n={len(nw)} | 정답 {len(nw)-len(nw_bad)} | 조작 오류 {nw_ctrl} ({nw_ctrl/len(nw):.1%}) | 그 외 오답 {len(nw_bad)-nw_ctrl}')
    for b in nw_bad: print(' ', b)
    record(dict(name='학습 후 / 해당 없음 신규물건', n=len(nw), exact=(len(nw)-len(nw_bad))/len(nw), fmt=1.0, misop=nw_ctrl/len(nw)))
else:
    print('eval_해당없음_신규물건.jsonl 없음 → 건너뜀')

## 11. STT 오인식 입력 평가 ("STT가 틀려도 LLM이 얼마나 맞히는가")
`eval_stt_aug.jsonl`(기존)과, 있으면 `eval_stt_dict_holdout.jsonl`(사전 적용 후 holdout)을 각각 평가합니다.

In [ ]:
def eval_stt(pat, title):
    p = find(pat)
    if not p: print(f'{pat} 없음 → 건너뜀'); return
    rows_ = load(p); preds_ = generate(rows_)
    n = len(rows_); em = sum(a == r['output'] for r, a in zip(rows_, preds_)); fmt = sum(bool(VALID.match(a)) for a in preds_)
    print(f'\n===== {title}: n={n}  정확 일치 {em/n:.1%} ({em}/{n}) | 형식 준수 {fmt/n:.1%}')
    bycat = collections.defaultdict(lambda: [0, 0])
    for r, a in zip(rows_, preds_): bycat[r['category']][0] += (a == r['output']); bycat[r['category']][1] += 1
    print('유형별:', {k: f'{a}/{b} ({a/b:.0%})' for k, (a, b) in sorted(bycat.items())})
    na_ = [(r, a) for r, a in zip(rows_, preds_) if r['output'] == '<six_2>()<six_end>']
    if na_: print('해당 없음 문장 조작 오류:', sum(bool(CTRL.search(a)) for r, a in na_), '/', len(na_))
    wrong = [(r.get('id'), r['category'], r['text'], r.get('gold_text', ''), r['output'], a) for r, a in zip(rows_, preds_) if a != r['output']]
    outp = f'{OUT}/stt_wrong_{pat.split(".")[0]}.csv'
    with open(outp, 'w', newline='', encoding='utf-8-sig') as f:
        w = csv.writer(f); w.writerow(['id', '분류', '입력 글자', '원래 정답 문장', '정답 호출', '모델 출력']); w.writerows(wrong)
    print('틀린 문장', len(wrong), '개 →', outp)
    for x in wrong[:15]: print(' ', x[1], '|', x[2], '→', x[5])
    record(dict(name=f'학습 후 / {title}', n=n, exact=em/n, fmt=fmt/n, misop=(sum(bool(CTRL.search(a)) for r, a in na_) / max(len(na_), 1))))

eval_stt('eval_stt_aug.jsonl', 'STT 오인식(기존 eval_stt_aug)')
eval_stt('eval_stt_dict_holdout.jsonl', 'STT 오인식(사전 적용 holdout)')

## 12. 신뢰도 구간 (즉시 실행 / 3초 확인 / 되묻기)
※ 90% / 70% 기준은 설계 가정치입니다.

In [ ]:
@torch.no_grad()
def seq_conf(text, pred):
    prompt = tok.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':text}],
                                     tokenize=False, add_generation_prompt=True, enable_thinking=False)
    p = tok(prompt, add_special_tokens=False)['input_ids']
    a = tok(pred + tok.eos_token, add_special_tokens=False)['input_ids']
    ids = torch.tensor([p + a], device='cuda')
    with torch.autocast('cuda', dtype=dtype):
        logits = model(input_ids=ids).logits[0].float()
    lp = torch.log_softmax(logits[len(p)-1:-1], dim=-1)
    tok_lp = lp.gather(1, torch.tensor(a, device='cuda').unsqueeze(1)).squeeze(1)
    return math.exp(tok_lp.sum().item()), math.exp(tok_lp.min().item())

def collect(rows_, name):
    preds_ = generate(rows_); out = []
    for r, q in zip(rows_, preds_):
        s, m = seq_conf(r['text'], q)
        out.append(dict(set=name, text=r['text'], gold=r['output'], pred=q, correct=(q == r['output']), seq_p=s, min_p=m))
    return out

_extra = find('eval_v3_해당없음_추가.jsonl'); extra = load(_extra) if _extra else []
data = collect(ev3, '20·30대 v3') + collect(ev_clean, '50대 v2') + collect(val[:200], '검증셋')
if extra: data += collect(extra, '해당 없음 추가')
df = pd.DataFrame(data)
df.to_csv(f'{ROOT}/runs/confidence_{MODEL_KEY}_{VERSION}.csv', index=False, encoding='utf-8-sig')
print('문장 수', len(df), '| 정확도', f'{df.correct.mean():.1%}')

for col in ['seq_p', 'min_p']:
    print(f'\n=== 신뢰도 = {col} ===')
    tab = []
    for t in [0.99, 0.95, 0.90, 0.80, 0.70, 0.50]:
        up = df[df[col] >= t]
        tab.append({'기준 이상': t, '문장 수': len(up), '비율': f'{len(up)/len(df):.1%}',
                    '그 안의 정답률': f'{up.correct.mean():.1%}' if len(up) else '-',
                    '걸러진 오답': f'{int((~df.correct).sum() - (~up.correct).sum())}/{int((~df.correct).sum())}'})
    display(pd.DataFrame(tab))
    hi, lo = 0.90, 0.70
    b = pd.DataFrame([
        {'구간': f'즉시 실행 (≥{hi:.0%})', '문장 수': int((df[col] >= hi).sum()), '정답률': df[df[col] >= hi].correct.mean()},
        {'구간': f'3초 확인 ({lo:.0%}~{hi:.0%})', '문장 수': int(((df[col] >= lo) & (df[col] < hi)).sum()), '정답률': df[(df[col] >= lo) & (df[col] < hi)].correct.mean()},
        {'구간': f'되묻기 (<{lo:.0%})', '문장 수': int((df[col] < lo).sum()), '정답률': df[df[col] < lo].correct.mean()}])
    b['정답률'] = b['정답률'].map(lambda x: '-' if x != x else f'{x:.1%}')
    display(b)

print('\n[틀린 문장과 신뢰도]')
display(df[~df.correct][['set', 'text', 'pred', 'seq_p', 'min_p']].round(3))

## 13. 직접 입력해서 확인 (시연)

In [ ]:
NAME = {1: '조명', 2: '에어컨', 3: '블라인드'}
ACT = {(0,1,1): '켜기', (0,1,2): '끄기', (1,1,1): '더 밝게', (1,1,2): '은은하게',
       (0,2,1): '켜기(24°C)', (0,2,2): '끄기', (1,2,1): '온도 올리기', (1,2,2): '온도 내리기',
       (1,2,3): '바람 세게', (1,2,4): '바람 약하게',
       (0,3,1): '올리기', (0,3,2): '내리기', (0,3,3): '멈추기', (0,3,4): '반쯤 내리기'}
def interpret(out):
    if '<six_2>' in out: return ['해당 없음 (안내만)']
    res = []
    for k, o, t in re.findall(r'<six_(\d)>\(object=(-?\d+)(?:,\s*type=(\d+))?\)', out):
        k, o = int(k), int(o)
        if o == -1: res.append('되묻기: 어떤 기기를 말씀하시는 건가요?')
        else: res.append(f"{NAME.get(o, '?')} {ACT.get((k, o, int(t or 0)), '?')}")
    return res or ['형식 오류 → 되묻기']
def demo(text):
    out = generate([{'text': text}], bs=1)[0]
    print('입력  :', text); print('모델  :', out)
    for i, a in enumerate(interpret(out), 1): print(f'실행{i} :', a)

for t in ['어두워 죽겠네 불 좀 키라', '블라인드 올리고 에어컨 좀 키라', '춥다 온도 좀 올리라', '에어컨 거프고 전등도 좀 꺼도',
          '가스불 꺼뿌라', '고마 꺼라', '덥다 아이가']:
    demo(t); print()

## 14. Llama vs Qwen 비교표
두 모델을 모두 학습한 뒤 실행합니다. 이전 버전(v1, v2, v2dict)과 이번 버전(v2dict2)을 한 표에 보여 줍니다.

In [ ]:
VERSIONS = ['v1', 'v2', 'v2dict', VERSION]
comp = []
for k in ['llama', 'qwen']:
    for v in dict.fromkeys(VERSIONS):
        p = f'{ROOT}/runs/lora-{k}/results.json' if v == 'v1' else f'{ROOT}/runs/lora-{k}-{v}/results.json'
        if not os.path.exists(p): print(k, v, '결과 없음 - 해당 모델/버전으로 먼저 실행하세요'); continue
        d = json.load(open(p, encoding='utf-8')); m = d['meta']; R = {r['name']: r for r in d['results']}
        g = lambda name, key: R[name][key] if name in R else float('nan')
        comp.append({'모델': m.get('model', k).split('/')[-1], '데이터': v, '학습 행수': m.get('train_rows', ''), '크기(B)': m.get('params_B'),
            '학습 전 50대 의도일치': g('학습 전 / 50대 평가', 'exact'), '학습 후 50대 의도일치': g('학습 후 / 50대 평가', 'exact'),
            '학습 후 val 의도일치': g('학습 후 / val', 'exact'), '형식 준수(50대)': g('학습 후 / 50대 평가', 'fmt'),
            '오작동률(50대)': g('학습 후 / 50대 평가', 'misop'),
            'STT 오인식 일치': g('학습 후 / STT 오인식(기존 eval_stt_aug)', 'exact'),
            'STT 오인식(사전 holdout) 일치': g('학습 후 / STT 오인식(사전 적용 holdout)', 'exact'),
            '해당없음 50대검수 조작오류': g('학습 후 / 해당 없음 50대검수', 'misop'), '해당없음 신규물건 조작오류': g('학습 후 / 해당 없음 신규물건', 'misop'),
            '학습 시간(분)': round(m.get('train_sec', 0) / 60, 1), '추론(초/문장)': m.get('gen_sec_per_sample')})
pd.options.display.float_format = '{:.3f}'.format
cdf = pd.DataFrame(comp); display(cdf)
cdf.to_csv(f'{ROOT}/runs/model_comparison.csv', index=False, encoding='utf-8-sig')

## 15. 완성 모델 저장 (양자화 단계용) — 가장 마지막에 실행
어댑터를 베이스 모델에 합쳐 저장합니다. **이 셀을 실행하면 모델이 합쳐진 상태가 되므로, 위의 평가를 모두 끝낸 뒤에 실행하세요.** (이후 평가를 다시 하려면 세션을 다시 시작하고 `RUN_TRAIN=False`로 실행)

In [ ]:
MERGED = f'{ROOT}/runs/merged-{MODEL_KEY}-{VERSION}'
merged = model.merge_and_unload()
merged.save_pretrained(MERGED, safe_serialization=True); tok.save_pretrained(MERGED)
print('합친 모델 저장:', MERGED)